# Embedding delle immagini con CLIP (Op. 29)

CLIP trasforma immagini e testi in vettori dello **stesso spazio** (512 numeri): un'immagine e una frase che la descrive hanno vettori vicini. Qui si calcola il vettore di ciascuna delle 41.460 immagini e si usa in tre modi (architettura-retrieval.md, §3):
1. **`labelConfidence`**: quanto un'immagine somiglia alle altre immagini dello stesso giocatore. Un'etichetta sbagliata (un altro giocatore, un segnaposto) dà un valore basso. Verifica automatica: le 268 immagini duplicate tra giocatori diversi devono avere valori più bassi delle altre.
2. **immagine migliore** della scheda giocatore: la non duplicata con `labelConfidence` più alta;
3. **ricerca di immagini da testo** (tipo R2): una frase passa nella parte testuale di CLIP e si cercano le immagini più vicine.

Codice in `src/wc_retrieval/embedding.py`. I vettori si salvano in `data/embeddings/` e si caricano in Neo4j con un indice vettoriale (cella 5, l'unica che scrive nel database).

**Prerequisiti (una sola volta), nel terminale del `.venv`:** `pip install sentence-transformers` (installa anche `torch` per CPU, circa 1–2 GB). Al primo avvio il modello CLIP (circa 600 MB) si scarica da Hugging Face.

**Tempi:** il calcolo (cella 3) gira su CPU e dura indicativamente 20–60 minuti; la cella 2 lo stima sul tuo PC. Il calcolo procede a lotti di 1.000 immagini salvati su disco: se si interrompe, rieseguendo la cella riparte dal primo lotto mancante.

In [ ]:
# [Cella 1] Setup: codice del retrieval, percorsi e tabella delle immagini (Op. 24)
import importlib
import sys
import time
from getpass import getpass
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Percorsi e codice del progetto (src/wc_retrieval)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
import wc_retrieval.embedding
import wc_retrieval.grafo
import wc_retrieval.modelli
import wc_retrieval.ricerca
for modulo in (wc_retrieval.grafo, wc_retrieval.ricerca, wc_retrieval.modelli, wc_retrieval.embedding):
    importlib.reload(modulo)
from wc_retrieval import embedding as emb
from wc_retrieval import modelli, ricerca
from wc_retrieval.grafo import BASE, Grafo

OUT = ROOT / "data" / "processed"
EMB = ROOT / "data" / "embeddings"                       # vettori (dati derivati, decisione R6)
EMB.mkdir(parents=True, exist_ok=True)
DIR4 = ROOT / "data" / "raw" / "dataset_4" / "Images" / "Images"

# Le 41.460 immagini del grafo (tabella del notebook 02)
img = pd.read_parquet(OUT / "image.parquet")
img["uri"] = BASE + "image/" + img["player_id"] + "/" + img["numero"]       # stesso URI dei nodi Image
# Ogni file deve esistere su disco prima del calcolo
presenti = img["filePath"].map(lambda p: (DIR4 / p).is_file())
print("immagini:", len(img), "| file presenti su disco:", presenti.sum(), "| giocatori:", img["player_id"].nunique())
assert presenti.all(), "Mancano dei file: controlla DIR4"

immagini: 41460 | file presenti su disco: 41460 | giocatori: 830


In [ ]:
# [Cella 2] Modello CLIP e stima del tempo di calcolo
t0 = time.time()
# CLIP ViT-B/32: trasforma immagini e testi in vettori di 512 numeri nello stesso spazio
clip = emb.carica_clip()
print(f"modello {emb.MODELLO_CLIP} caricato in {time.time() - t0:.0f} s")

# Prova su 64 immagini per stimare il tempo totale
prova = [emb.apri_immagine(DIR4 / p) for p in img["filePath"][:64]]
t0 = time.time()
v = clip.encode(prova, batch_size=32, convert_to_numpy=True, normalize_embeddings=True)
velocita = len(prova) / (time.time() - t0)
print("vettori di prova:", v.shape, "| norma:", np.linalg.norm(v, axis=1).round(3)[:3], "(normalizzati)")
print(f"{velocita:.1f} immagini al secondo → stima per tutte: {len(img) / velocita / 60:.0f} minuti")

modello clip-ViT-B-32 caricato in 169 s
vettori di prova: (64, 512) | norma: [1. 1. 1.] (normalizzati)
3.3 immagini al secondo → stima per tutte: 209 minuti


In [ ]:
# [Cella 3] Calcolo degli embedding di tutte le immagini (lungo; si può interrompere e riprendere)
# Calcolo a lotti di 1.000 salvati su disco: se si interrompe, riparte dal primo lotto mancante
vettori, errori = emb.calcola_embedding_immagini(img[["uri", "filePath"]], DIR4, EMB / "lotti_immagini",
                                                 clip, lotto=1000, batch=32)
print("embedding:", len(vettori), "| file non aperti:", len(errori))
if len(errori):
    print(errori.to_string(index=False))

embedding: 41460 | file non aperti: 0


## labelConfidence
Per ogni immagine: similarità coseno con il centro delle **altre** immagini dello stesso giocatore (si esclude l'immagine stessa, così non "vota" per sé). Valori alti = l'immagine somiglia alle altre della sua cartella. Il confronto con le immagini duplicate tra giocatori diversi (`hasDuplicateContent`, Op. 24), che sono sicuramente sbagliate per almeno un giocatore, verifica che la misura funzioni. L'**AUC** è la probabilità che un'immagine non duplicata abbia una confidenza più alta di una duplicata: 0,5 = la misura non distingue, 1 = separazione perfetta.

In [ ]:
# [Cella 4] labelConfidence, verifica sui duplicati e salvataggio
tab = img[["uri", "player_id", "player_name", "team_name", "filePath", "md5", "hasDuplicateContent"]] \
    .merge(vettori, on="uri", validate="one_to_one")
# Coseno tra l'immagine e il centro delle ALTRE immagini dello stesso giocatore
tab["labelConfidence"] = emb.confidenza_etichette(tab)
tab[["uri", "player_id", "embedding", "labelConfidence"]].to_parquet(EMB / "immagini_clip.parquet", index=False)
print("salvato:", EMB / "immagini_clip.parquet", "|", len(tab), "immagini")

# Verifica: le immagini duplicate tra giocatori diversi devono avere confidenza più bassa
c = tab["labelConfidence"]
print("labelConfidence:", c.describe().round(3).to_dict())
dup, ok = tab[tab["hasDuplicateContent"]], tab[~tab["hasDuplicateContent"]]
print(f"media: non duplicate {ok['labelConfidence'].mean():.3f} | duplicate {dup['labelConfidence'].mean():.3f}")
# AUC: probabilità che una non duplicata superi una duplicata (0,5 = a caso, 1 = perfetto)
print(f"AUC (non duplicate più alte delle duplicate): {emb.auc(ok['labelConfidence'], dup['labelConfidence']):.3f}")
soglia = ok["labelConfidence"].quantile(0.05)
print(f"duplicate sotto il 5° percentile delle non duplicate ({soglia:.3f}): "
      f"{(dup['labelConfidence'] < soglia).sum()} su {len(dup)}")
segnaposto = tab[tab["md5"] == "cb5eff96a0999c67c66569081068aa7c"]           # lo stesso file in 32 cartelle (Op. 24)
print(f"file presente in 32 cartelle: confidenza media {segnaposto['labelConfidence'].mean():.3f}, "
      f"massima {segnaposto['labelConfidence'].max():.3f}")
per_giocatore = tab.groupby("player_name")["labelConfidence"].median().sort_values()
print("giocatori con la confidenza mediana più bassa:", per_giocatore.head(5).round(3).to_dict())

salvato: <progetto>\data\embeddings\immagini_clip.parquet | 41460 immagini
labelConfidence: {'count': 41460.0, 'mean': 0.812, 'std': 0.082, 'min': 0.091, '25%': 0.776, '50%': 0.829, '75%': 0.868, 'max': 0.974}
media: non duplicate 0.812 | duplicate 0.732
AUC (non duplicate più alte delle duplicate): 0.745
duplicate sotto il 5° percentile delle non duplicate (0.651): 68 su 268
file presente in 32 cartelle: confidenza media 0.635, massima 0.693
giocatori con la confidenza mediana più bassa: {'Douglas López': 0.637, 'Khalid Muneer': 0.655, 'Joel King': 0.663, 'Kevin Rodríguez': 0.672, 'Yousef Hassan': 0.679}


## Caricamento in Neo4j
Scrive `embedding` (512 numeri) e `labelConfidence` sui 41.460 nodi Image e crea l'indice vettoriale `image_clip` (similarità coseno, **senza quantizzazione**: dalla 5.23 Neo4j comprime di default i vettori dell'indice, e con punteggi CLIP molto vicini tra loro la compressione altera punteggi e ordine; l'indice viene cancellato e ricreato). È l'unica cella che scrive nel database. Se il grafo viene ricostruito, basta rieseguire le celle 1 e questa, leggendo `immagini_clip.parquet` (senza ricalcolare gli embedding).

In [ ]:
# [Cella 5] Caricamento in Neo4j (SCRIVE): embedding, labelConfidence, indice vettoriale
g = Grafo(getpass("Password Neo4j: "))
salvati = pd.read_parquet(EMB / "immagini_clip.parquet")      # si rilegge dal file: funziona anche dopo un riavvio
t0 = time.time()
# Scrive embedding e labelConfidence sui nodi Image, a lotti
print("nodi aggiornati:", emb.carica_embedding_immagini(g, salvati), f"in {time.time() - t0:.0f} s")
# Indice vettoriale coseno senza quantizzazione (R9): la compressione alterava l'ordine
emb.crea_indice_vettoriale(g, emb.INDICE_IMMAGINI, "Image", "embedding", emb.DIMENSIONI_CLIP)
# Controllo: stato dell'indice e nodi aggiornati
print(g.leggi("""
    SHOW VECTOR INDEXES YIELD name, state, labelsOrTypes, properties, options
    RETURN name AS indice, state AS stato, labelsOrTypes AS classe, properties AS proprieta,
           options.indexConfig['vector.dimensions'] AS dimensioni,
           options.indexConfig['vector.similarity_function'] AS similarita,
           options.indexConfig['vector.quantization.enabled'] AS quantizzazione""").to_string(index=False))
print(g.leggi("""
    MATCH (i:Image)
    RETURN count(i) AS immagini, count(i.embedding) AS con_embedding, count(i.labelConfidence) AS con_confidenza,
           round(avg(i.labelConfidence), 3) AS confidenza_media""").to_string(index=False))

nodi aggiornati: 41460 in 142 s
    indice  stato  classe   proprieta  dimensioni similarita  quantizzazione
image_clip ONLINE [Image] [embedding]         512     COSINE           False
 immagini  con_embedding  con_confidenza  confidenza_media
    41460          41460           41460             0.812
